In [ ]:
import pygame
import random

# Inicializar o Pygame
pygame.init()

# Cores
BRANCO = (255, 255, 255)
PRETO = (0, 0, 0)
CINZA = (200, 200, 200)
AZUL = (50, 150, 255)
VERDE = (50, 200, 50)
VERMELHO = (255, 50, 50)

# Configurações da Tela
LARGURA = 540
ALTURA = 600
TELA = pygame.display.set_mode((LARGURA, ALTURA))
pygame.display.set_caption("Sudoku com Pygame")
FONTE_NUMERO = pygame.font.SysFont("comicsans", 40)
FONTE_BOTAO = pygame.font.SysFont("comicsans", 25)

# Lógica do Sudoku
def eh_valido(tabuleiro, linha, coluna, num):
    # Checar linha
    for x in range(9):
        if tabuleiro[linha][x] == num:
            return False

    # Checar coluna
    for x in range(9):
        if tabuleiro[x][coluna] == num:
            return False

    # Checar quadrante 3x3
    inicio_linha = linha - linha % 3
    inicio_coluna = coluna - coluna % 3
    for i in range(3):
        for j in range(3):
            if tabuleiro[i + inicio_linha][j + inicio_coluna] == num:
                return False

    return True

def resolver_tabuleiro(tabuleiro):
    for linha in range(9):
        for coluna in range(9):
            if tabuleiro[linha][coluna] == 0:
                for num in range(1, 10):
                    if eh_valido(tabuleiro, linha, coluna, num):
                        tabuleiro[linha][coluna] = num
                        if resolver_tabuleiro(tabuleiro):
                            return True
                        tabuleiro[linha][coluna] = 0
                return False
    return True

def gerar_tabuleiro_completo():
    tabuleiro = [[0 for _ in range(9)] for _ in range(9)]
    
    # Preencher a diagonal principal (quadrantes 3x3) para gerar uma base válida
    for i in range(0, 9, 3):
        numeros = list(range(1, 10))
        random.shuffle(numeros)
        idx = 0
        for linha in range(3):
            for coluna in range(3):
                tabuleiro[i + linha][i + coluna] = numeros[idx]
                idx += 1
                
    # Resolver o resto usando backtracking
    resolver_tabuleiro(tabuleiro)
    return tabuleiro

def gerar_puzzle(dificuldade=20):
    tabuleiro_completo = gerar_tabuleiro_completo()
    puzzle = [linha[:] for linha in tabuleiro_completo]
    
    # Remover células aleatoriamente para criar o jogo
    celulas_para_remover = dificuldade
    while celulas_para_remover > 0:
        linha = random.randint(0, 8)
        coluna = random.randint(0, 8)
        if puzzle[linha][coluna] != 0:
            puzzle[linha][coluna] = 0
            celulas_para_remover -= 1
            
    return puzzle, tabuleiro_completo

# Interface Gráfica
def desenhar_grade(tabuleiro, selecionado):
    # Desenhar linhas e colunas
    for i in range(10):
        espessura = 4 if i % 3 == 0 else 1
        pygame.draw.line(TELA, PRETO, (i * 60, 0), (i * 60, 540), espessura)
        pygame.draw.line(TELA, PRETO, (0, i * 60), (LARGURA, i * 60), espessura)

    # Desenhar números
    for linha in range(9):
        for coluna in range(9):
            valor = tabuleiro[linha][coluna]
            if valor != 0:
                texto = FONTE_NUMERO.render(str(valor), 1, PRETO)
                TELA.blit(texto, (coluna * 60 + 20, linha * 60 + 15))

    # Desenhar retângulo da célula selecionada
    if selecionado:
        linha, coluna = selecionado
        pygame.draw.rect(TELA, AZUL, (coluna * 60, linha * 60, 60, 60), 3)

def desenhar_botoes():
    pygame.draw.rect(TELA, CINZA, (150, 550, 100, 40))
    texto_resolver = FONTE_BOTAO.render("Resolver", 1, PRETO)
    TELA.blit(texto_resolver, (160, 555))

    pygame.draw.rect(TELA, CINZA, (300, 550, 100, 40))
    texto_novo = FONTE_BOTAO.render("Novo", 1, PRETO)
    TELA.blit(texto_novo, (320, 555))

def main():
    executando = True
    puzzle, resolvido = gerar_puzzle(40)
    selecionado = None
    
    while executando:
        TELA.fill(BRANCO)
        desenhar_grade(puzzle, selecionado)
        desenhar_botoes()

        for evento in pygame.event.get():
            if evento.type == pygame.QUIT:
                executando = False
                
            elif evento.type == pygame.MOUSEBUTTONDOWN:
                pos = pygame.mouse.get_pos()
                
                # Clicar na grade
                if pos[1] < 540:
                    coluna = pos[0] // 60
                    linha = pos[1] // 60
                    selecionado = (linha, coluna)
                    
                # Clicar nos botões
                elif 150 <= pos[0] <= 250 and 550 <= pos[1] <= 590:
                    puzzle = [linha[:] for linha in resolvido] # Revelar a solução
                elif 300 <= pos[0] <= 400 and 550 <= pos[1] <= 590:
                    puzzle, resolvido = gerar_puzzle(40) # Gerar novo jogo
                    selecionado = None

            elif evento.type == pygame.KEYDOWN:
                if selecionado and puzzle[selecionado[0]][selecionado[1]] == 0:
                    valor = None
                    if evento.key in [pygame.K_1, pygame.K_KP1]: valor = 1
                    elif evento.key in [pygame.K_2, pygame.K_KP2]: valor = 2
                    elif evento.key in [pygame.K_3, pygame.K_KP3]: valor = 3
                    elif evento.key in [pygame.K_4, pygame.K_KP4]: valor = 4
                    elif evento.key in [pygame.K_5, pygame.K_KP5]: valor = 5
                    elif evento.key in [pygame.K_6, pygame.K_KP6]: valor = 6
                    elif evento.key in [pygame.K_7, pygame.K_KP7]: valor = 7
                    elif evento.key in [pygame.K_8, pygame.K_KP8]: valor = 8
                    elif evento.key in [pygame.K_9, pygame.K_KP9]: valor = 9
                    
                    if valor:
                        linha, coluna = selecionado
                        # Confere se a jogada respeita as regras do Sudoku
                        if eh_valido(puzzle, linha, coluna, valor):
                            puzzle[linha][coluna] = valor
                        else:
                            print("Jogada inválida!")

        pygame.display.flip()

    pygame.quit()

if __name__ == "__main__":
    main()